# Chapter 6 — Context Windows: Learning from More Than One Token

**Series:** LM from First Principles

Chapter 1 built a bigram language model: given one token, predict the next. The model is fast and simple, but it has a hard limitation — it can only see **one** preceding token. It cannot know that "New" likely follows "of" because earlier in the sequence the phrase started with "City".

This chapter answers one question:

> What changes when the model can use several preceding tokens instead of only one?

We extend the bigram to a **fixed-context-window model** that takes the last k tokens as input. We train it for k = 1, 2, 3, 4, 6, 8, measure what changes, and then ask what the model still cannot do — which motivates attention in Chapter 7.

**What you will build**

- A `FixedWindowModel` that concatenates k token embeddings and feeds them to a linear head
- A training sweep across six context lengths on tinyshakespeare
- Comparison tables: validation loss, perplexity, parameter count, training time
- Per-character surprisal analysis: which characters benefit most from longer context
- An n-gram bridge showing the conceptual link between count-based and neural models


## Setup — Same corpus, same vocabulary

We use tinyshakespeare (1 MB of Shakespeare plays) with a character-level vocabulary of 65 tokens. This is the same corpus as Chapter 5 so the results are directly comparable.


In [18]:
# ── imports ───────────────────────────────────────────────────────────────────
import torch                    # PyTorch: the deep learning framework used throughout the book
import torch.nn as nn           # nn.Module, nn.Embedding, nn.Linear — the building blocks
import torch.nn.functional as F # stateless functions: softmax, cross_entropy, log_softmax
import time                     # time.time() to measure wall-clock training duration

# ── load the raw corpus ───────────────────────────────────────────────────────
# tinyshakespeare.txt is ~1 MB of Shakespeare plays.
# It is the same corpus used in Chapter 5 so results are directly comparable.
with open('tinyshakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()   # one long Python string, ~1 million characters

# ── build the character-level vocabulary ──────────────────────────────────────
# set(text)    — all unique characters that appear anywhere in the corpus
# sorted(...)  — put them in a consistent order so IDs are stable across runs
# len(chars)   — how many unique characters = our vocabulary size
chars      = sorted(set(text))
vocab_size = len(chars)   # 65 for tinyshakespeare

# stoi (string-to-integer): maps each character to a unique integer ID 0..64
# itos (integer-to-string): reverse lookup — maps an integer ID back to its character
# These two dicts are used throughout: stoi to encode input, itos to decode output.
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

print(f"Corpus  : {len(text):,} characters")
print(f"Vocab   : {vocab_size} unique tokens")
print(f"Example : {repr(text[:80])}")

# ── encode the full corpus as a 1-D tensor of integer token IDs ───────────────
# We convert every character in text to its integer ID using stoi.
# dtype=torch.long is required — embedding tables index by integer, not by float.
# data shape: (len(text),) — one integer per character in the corpus.
data  = torch.tensor([stoi[c] for c in text], dtype=torch.long)

# ── train / validation split (90% train, 10% val) ────────────────────────────
# split is the index that divides the first 90% from the last 10%.
# train_data: used to update model weights during training (the model SEES this).
# val_data:   held out entirely — never used for weight updates.
#             Measuring loss on val_data tells us whether the model generalises
#             or just memorised the training examples.
split      = int(0.9 * len(data))
train_data = data[:split]
val_data   = data[split:]

print(f"\nTrain tokens: {len(train_data):,}")
print(f"Val tokens  : {len(val_data):,}")

# All models in this chapter are small enough to train on CPU.
device = 'cpu'


Corpus  : 1,115,394 characters
Vocab   : 65 unique tokens
Example : 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.'

Train tokens: 1,003,854
Val tokens  : 111,540


---

## Part 1 — From One Token to k Tokens

We start by revisiting what the bigram model does, then generalise it to arbitrary context length k.


## Section 1 — The bigram model is a fixed window of size 1

Chapter 1 trained a `BigramModel`: an `nn.Embedding(vocab_size, vocab_size)` where each row is the logit vector for the next token given that input token.

That is a fixed window of size **k = 1**. The model sees exactly one preceding token and ignores everything earlier.

```
Input  :  ... x_{t-2}  x_{t-1}  [x_t]  →  predict x_{t+1}
                                  ↑
                       only this token used
```

The fundamental limitation: no matter how many tokens precede the current position, the bigram model discards all but the last one.


In [19]:
# ── BigramModel: restated from Chapter 1 ─────────────────────────────────────
# We retrain it here so the k=1 comparison uses the same training setup as k=2..8.
# This makes any difference in results attributable to context length, not setup.

class BigramModel(nn.Module):
    # The simplest possible language model: given ONE token, predict the next.
    #
    # Architecture: a single embedding table of shape (vocab_size, vocab_size).
    # Each row i is the logit vector for predicting the next token when the input is i.
    # The embedding lookup IS the prediction — embedding and head are fused into one table.
    # This is conceptually identical to FixedWindowModel(k=1, embed_dim=vocab_size).

    def __init__(self, vocab_size):
        super().__init__()

        # EMBEDDING TABLE — shape (65, 65)
        # Row i = the 65 logits (one per vocab token) the model predicts after seeing token i.
        # There is no separate linear head — the embedding output is the final prediction.
        # Total parameters: vocab_size × vocab_size = 65 × 65 = 4,225.
        self.emb = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x):
        # x shape: (batch,) — one token ID per example; context length = 1.
        # self.emb(x) looks up one row per token ID in the embedding table.
        # Output shape: (batch, vocab_size) = (batch, 65) — raw logits, not probabilities.
        # Apply softmax to convert to a probability distribution over the vocabulary.
        return self.emb(x)   # shape: (batch, vocab_size)

# ── get_batch_bigram: sample a batch for the k=1 model ───────────────────────
def get_batch_bigram(data, batch_size):
    # Sample batch_size random positions from the data tensor.
    # x = the token at position i     (context: one token)
    # y = the token at position i + 1 (target: what we want to predict)
    # torch.randint(0, len(data)-1, ...) ensures i+1 is always a valid index.
    ix = torch.randint(0, len(data) - 1, (batch_size,))
    x  = data[ix]       # shape: (batch_size,)  — the context tokens
    y  = data[ix + 1]   # shape: (batch_size,)  — the target tokens
    return x, y

# ── count and describe the model ──────────────────────────────────────────────
bm = BigramModel(vocab_size)
print(f"BigramModel parameters: {sum(p.numel() for p in bm.parameters()):,}")
print(f"  = vocab_size × vocab_size = {vocab_size} × {vocab_size}")
print()
print("Input  : one token ID  (batch,)")
print("Lookup : embedding row  → (batch, vocab_size) logits")
print("Context: exactly k = 1 preceding token")


BigramModel parameters: 4,225
  = vocab_size × vocab_size = 65 × 65

Input  : one token ID  (batch,)
Lookup : embedding row  → (batch, vocab_size) logits
Context: exactly k = 1 preceding token


## Section 2 — A fixed-window model for any k

To use k preceding tokens we:

1. **Embed** each of the k tokens independently: k vectors of size `embed_dim`
2. **Concatenate** them into one vector of size `k × embed_dim`
3. **Project** with a linear layer to `vocab_size` logits

```
x_{t-k}  →  emb  →  v_1  ┐
x_{t-k+1}→  emb  →  v_2  ├─ concat → [v_1 | v_2 | … | v_k] → Linear → logits
   …                  …   │
x_{t-1}  →  emb  →  v_k  ┘
```

Every position has its **own slice of the weight matrix** in the linear head. Position −k always contributes through the first `embed_dim` columns; position −1 always contributes through the last `embed_dim` columns. The weights are fixed to positions, not to content — this is the limitation we will examine in Section 9.

**Parameter count**

| Component | Size |
|-----------|------|
| Embedding table | `vocab_size × embed_dim` |
| Linear head | `(k × embed_dim) × vocab_size + vocab_size` |
| **Total** | `vocab_size × embed_dim + k × embed_dim × vocab_size + vocab_size` |


In [20]:
# ── global embedding dimension ───────────────────────────────────────────────
EMBED_DIM = 64
# 64 is small enough to train on CPU in seconds, large enough to learn
# useful character patterns. Every model in this chapter shares this value.

class FixedWindowModel(nn.Module):
    # PURPOSE: given the k most recent token IDs, predict the next token.
    #
    # ARCHITECTURE (two components):
    #
    #   1. INPUT EMBEDDING TABLE — nn.Embedding(vocab_size, embed_dim)
    #      Shape: (vocab_size, embed_dim) = (65, 64).
    #      Acts as a lookup table: row i is the learned vector for token i.
    #      Maps each integer token ID → a dense float vector of size embed_dim.
    #      These vectors are learned during training; they start random.
    #
    #   2. LINEAR HEAD — nn.Linear(k * embed_dim, vocab_size)
    #      Shape of weight matrix: (vocab_size, k * embed_dim) = (65, k*64).
    #      Takes the flattened context embedding and projects it to vocab_size logits.
    #      One output logit per vocabulary token = one score per possible next character.
    #
    # TOTAL PARAMETERS:
    #   Embedding : vocab_size × embed_dim      = 65 × 64    = 4,160
    #   Head w    : k × embed_dim × vocab_size  = k × 64 × 65
    #   Head bias :                               vocab_size  =    65
    #   Total grows linearly with k — only the head changes size.

    def __init__(self, vocab_size, k, embed_dim=EMBED_DIM):
        super().__init__()
        self.k = k   # context length — how many preceding tokens to use

        # INPUT EMBEDDING TABLE — shape (vocab_size, embed_dim) = (65, 64).
        # nn.Embedding is essentially a matrix; indexing with a token ID
        # returns the corresponding row (the token's learned vector).
        self.embed = nn.Embedding(vocab_size, embed_dim)

        # LINEAR HEAD — shape maps (k * embed_dim) → vocab_size.
        # This is the prediction layer: it takes the flattened context embedding
        # and produces one logit per vocabulary token.
        self.head = nn.Linear(k * embed_dim, vocab_size)

    def forward(self, x):
        # x shape: (batch, k) — each row is one context window of k token IDs.

        # STEP 1 — embedding lookup.
        # For each of the k positions, look up the corresponding row in the
        # embedding table. Returns a 3-D tensor — one vector per position per example.
        emb = self.embed(x)              # shape: (batch, k, embed_dim) = (batch, k, 64)

        # STEP 2 — flatten positions into one long vector per example.
        # emb.size(0) = batch size. -1 tells PyTorch to infer the second dimension.
        # (batch, k, 64) → (batch, k * 64).
        # WHY: the linear head expects a 2-D input. The flattening encodes position
        # implicitly: slots 0..63 always correspond to the oldest token, 64..127 to
        # the next, etc. The head learns different weights for each positional slot.
        emb = emb.view(emb.size(0), -1)  # shape: (batch, k * embed_dim)

        # STEP 3 — linear head: project to vocabulary scores.
        # W @ emb + b  where W has shape (vocab_size, k * embed_dim).
        # Each of the vocab_size output values is a logit (raw score) for that token.
        # Higher logit → model thinks that token is more likely to come next.
        return self.head(emb)            # shape: (batch, vocab_size) = (batch, 65)

# ── parameter count table ─────────────────────────────────────────────────────
# Show exactly how many numbers each model learns as k grows.
# The embedding table is CONSTANT (only depends on vocab_size and embed_dim).
# Only the head weight matrix grows — by embed_dim × vocab_size per unit of k.
def param_count(k, vocab_size=vocab_size, embed_dim=EMBED_DIM):
    emb_params  = vocab_size * embed_dim              # embedding table
    head_params = k * embed_dim * vocab_size + vocab_size  # head weights + bias
    return emb_params + head_params

print(f"{'k':>4}  {'parameters':>12}  {'head input dim':>14}")
print(f"{'─'*4}  {'─'*12}  {'─'*14}")
for k in [1, 2, 3, 4, 6, 8]:
    print(f"{k:>4}  {param_count(k):>12,}  {k * EMBED_DIM:>14}")
print()
print(f"Bigram (k=1, embed_dim=vocab_size fused): {vocab_size * vocab_size:,}")
print("  The bigram model fuses embedding table and head into one (65×65) matrix.")


   k    parameters  head input dim
────  ────────────  ──────────────
   1         8,385              64
   2        12,545             128
   3        16,705             192
   4        20,865             256
   6        29,185             384
   8        37,505             512

Bigram (k=1, embed_dim=vocab_size fused): 4,225
  The bigram model fuses embedding table and head into one (65×65) matrix.


In [21]:
# ── concrete walkthrough: what happens to one context window ─────────────────
#
# We trace a single (context → logits) pass step by step, printing shapes at each stage.
# This makes the abstract architecture tangible before we start training.

demo_text = "First Citizen:\nBefore we proceed"
k_demo    = 3
context   = demo_text[-k_demo:]           # last 3 characters of demo_text
ctx_ids   = [stoi[c] for c in context]    # encode each character as an integer token ID

print(f"Context string : {repr(context)}")
print(f"Context IDs    : {ctx_ids}   (integers 0-64)")
print()

# STEP 1 — build the input tensor.
# Shape: (1, k) — batch size of 1 (one context window), k token IDs wide.
x = torch.tensor([ctx_ids])   # shape: (1, 3)
print(f"Input tensor shape : {list(x.shape)}  — (batch=1, k={k_demo})")

# STEP 2 — create a k=3 model and run a forward pass (weights are random — not yet trained).
demo_model = FixedWindowModel(vocab_size, k_demo)
demo_model.eval()   # disable dropout etc. (not used here; good habit)

with torch.no_grad():   # no gradient needed — just inspecting shapes

    # STEP 2a — embedding lookup.
    emb = demo_model.embed(x)    # shape: (1, k, embed_dim) = (1, 3, 64)
    print(f"After embedding    : {list(emb.shape)}  — one 64-d vector per context position")

    # STEP 2b — flatten.
    flat = emb.view(emb.size(0), -1)   # shape: (1, k * embed_dim) = (1, 192)
    print(f"After flatten      : {list(flat.shape)}  — three position vectors concatenated")

    # STEP 2c — linear head.
    logits = demo_model.head(flat)     # shape: (1, vocab_size) = (1, 65)
    print(f"After head         : {list(logits.shape)}  — one logit per vocabulary token")

# STEP 3 — convert logits to probabilities with softmax.
# softmax normalises the 65 raw scores so they sum to 1.
# The result is a valid probability distribution over the vocabulary.
probs = torch.softmax(logits[0], dim=-1)   # [0] selects the single batch row; shape: (65,)
print()
print(f"Logit range : [{logits.min().item():.3f}, {logits.max().item():.3f}]  (raw scores, can be negative)")
print(f"Prob range  : [{probs.min().item():.4f}, {probs.max().item():.4f}]  (sums to 1.0)")
print()

# Top-5 predictions from the untrained model (random weights → near-uniform distribution).
top = torch.topk(probs, 5)
print("Top-5 predicted next characters (untrained — should be near-uniform):")
for prob, idx in zip(top.values, top.indices):
    print(f"  {repr(itos[idx.item()]):<6}  p = {prob.item():.4f}")

print()
print("After training, one character will dominate for predictable contexts.")


Context string : 'eed'
Context IDs    : [43, 43, 42]   (integers 0-64)

Input tensor shape : [1, 3]  — (batch=1, k=3)
After embedding    : [1, 3, 64]  — one 64-d vector per context position
After flatten      : [1, 192]  — three position vectors concatenated
After head         : [1, 65]  — one logit per vocabulary token

Logit range : [-1.326, 1.225]  (raw scores, can be negative)
Prob range  : [0.0040, 0.0509]  (sums to 1.0)

Top-5 predicted next characters (untrained — should be near-uniform):
  's'     p = 0.0509
  'e'     p = 0.0375
  'P'     p = 0.0358
  '$'     p = 0.0352
  'T'     p = 0.0336

After training, one character will dominate for predictable contexts.


---

## Part 2 — Training Across Context Lengths

We now train one `FixedWindowModel` for each k ∈ {1, 2, 3, 4, 6, 8} and record four metrics: validation loss, perplexity, parameter count, and wall-clock training time.


## Section 3 — Training k = 1, 2, 3, 4, 6, 8

**Setup** — same for all runs:

| Setting | Value |
|---------|-------|
| Embedding dim | 64 |
| Optimizer | AdamW, lr = 1e-3 |
| Batch size | 64 |
| Steps | 3 000 |
| Val evaluation | every 500 steps (batch of 512) |

Training 6 models takes roughly 15–30 seconds on CPU.


In [ ]:
# ── get_batch: sample random (context, target) pairs ─────────────────────────
def get_batch(data, k, batch_size):
    # PURPOSE: produce a training batch of (context windows, targets) for context length k.
    #
    # HOW IT WORKS:
    #   1. Sample batch_size random positions ix from the data tensor.
    #   2. For each position i, the context is the k tokens immediately before i: data[i-k:i].
    #   3. The target is the token at position i: data[i].
    #
    # BOUNDS:
    #   Lower bound = k:            ensures data[i-k:i] never goes before the start of data.
    #   Upper bound = len(data)-1:  ensures data[i] (the target) is a valid index.
    ix = torch.randint(k, len(data) - 1, (batch_size,))

    # Build the context tensor by stacking batch_size slices of length k.
    # data[i-k:i] is a 1-D tensor of k token IDs.
    # torch.stack([...]) turns a list of 1-D tensors into a 2-D tensor.
    # x shape: (batch_size, k) — each row is one context window.
    x = torch.stack([data[i - k : i] for i in ix])

    # y shape: (batch_size,) — the single target token for each context.
    y = data[ix]
    return x, y

# ── eval_loss: estimate validation loss on n_samples random positions ─────────
def eval_loss(model, data, k, n_samples=512):
    # Switch to eval mode: disables dropout/batchnorm (not used here, good habit).
    model.eval()
    # torch.no_grad(): skip building the computation graph — measuring, not training.
    # This saves memory and speeds up the evaluation pass.
    with torch.no_grad():
        x, y = get_batch(data, k, n_samples)
        # F.cross_entropy: applies log-softmax to logits internally,
        # then computes -log(p(correct token)) for each example and returns the mean.
        # Lower cross-entropy = model assigns higher probability to the correct tokens.
        return F.cross_entropy(model(x), y).item()

# ── training hyperparameters ──────────────────────────────────────────────────
N_STEPS    = 3000   # number of gradient update steps per model
BATCH_SIZE = 64     # examples per update — larger = more stable gradient, slower per step
LR         = 1e-3   # learning rate: how large a weight update per step (1e-3 = 0.001)
K_VALUES   = [1, 2, 3, 4, 6, 8, 10, 15, 20]   # context lengths we train and compare

# results dict: after training, stores everything needed for comparison.
# Structure: {k: {model, val_loss, perplexity, params, train_sec, loss_curve}}
results = {}

# Fix the random seed ONCE before the loop.
# All 6 models start from the same random initialisation state.
# Any difference in final val_loss comes from k (context length), not from luck.
torch.manual_seed(42)

# ── training loop: one model per context length ───────────────────────────────
for k in K_VALUES:

    # CREATE MODEL — a fresh FixedWindowModel for this k.
    # Each model has its own embedding table and head; no weights are shared across k values.
    model = FixedWindowModel(vocab_size, k).to(device)

    # CREATE OPTIMISER — AdamW (Adam with weight decay).
    # model.parameters(): iterator over all learnable tensors (embedding table + head).
    # lr=LR: step size — how far to move weights in the gradient direction each update.
    # Weight decay: small L2 penalty that prevents weights from growing too large (regularisation).
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    loss_curve = []          # will hold (step, val_loss) tuples — one entry per 500 steps
    t0         = time.time() # record start time to measure total training duration

    # ── inner training loop ───────────────────────────────────────────────────
    for step in range(N_STEPS):
        model.train()   # switch to training mode — enables gradient computation

        # STEP 1 — sample a fresh random batch from the training data.
        # A new batch is drawn every step — the model never repeats the exact same batch.
        x, y = get_batch(train_data, k, BATCH_SIZE)

        # STEP 2 — forward pass: compute logits and the training loss.
        # logits shape: (BATCH_SIZE, vocab_size) — one score per vocab token per example.
        # F.cross_entropy compares logits to true targets y and returns the mean loss.
        # Loss is high when the model puts low probability on the correct token.
        logits = model(x)
        loss   = F.cross_entropy(logits, y)

        # STEP 3 — backward pass: compute gradients and update weights.
        optimizer.zero_grad()   # MUST clear old gradients — they accumulate if not cleared
        loss.backward()         # compute d(loss)/d(weight) for every parameter via backprop
        optimizer.step()        # w = w - lr * gradient  for every weight in the model

        # ── periodic validation check ─────────────────────────────────────────
        # Every 500 steps, evaluate on val_data (not train_data) to track generalisation.
        # We do NOT update weights here — this is measurement only.
        if (step + 1) % 500 == 0:
            vl = eval_loss(model, val_data, k)
            loss_curve.append((step + 1, vl))

    # ── post-training metrics ─────────────────────────────────────────────────
    elapsed = time.time() - t0   # total seconds elapsed during training

    # Final validation loss: use 2048 samples for a more stable estimate than 512.
    val_loss = eval_loss(model, val_data, k, n_samples=2048)

    # Perplexity = exp(val_loss).
    # Interpretation: if perplexity = P, the model is as uncertain as choosing uniformly
    # among P equally likely tokens at every position.
    # Lower perplexity = better model. Random guess over 65 tokens → perplexity = 65.
    ppl = torch.exp(torch.tensor(val_loss)).item()

    # Count every learnable number in the model (embedding table + head weights + bias).
    params = sum(p.numel() for p in model.parameters())

    # Store all results for the comparison cells that follow.
    results[k] = {
        'model'      : model,       # the trained model object
        'val_loss'   : val_loss,    # final validation cross-entropy loss
        'perplexity' : ppl,         # exp(val_loss) — easier to interpret than raw loss
        'params'     : params,      # total learnable parameters in this model
        'train_sec'  : elapsed,     # wall-clock training time in seconds
        'loss_curve' : loss_curve,  # list of (step, val_loss) for plotting convergence
    }
    print(f"k={k}  val_loss={val_loss:.4f}  ppl={ppl:.2f}  "
          f"params={params:,}  time={elapsed:.1f}s")


k=1  val_loss=2.5273  ppl=12.52  params=8,385  time=3.8s
k=2  val_loss=2.3815  ppl=10.82  params=12,545  time=3.4s
k=3  val_loss=2.2623  ppl=9.61  params=16,705  time=4.0s
k=4  val_loss=2.2770  ppl=9.75  params=20,865  time=3.5s
k=6  val_loss=2.2347  ppl=9.34  params=29,185  time=3.5s
k=8  val_loss=2.3219  ppl=10.19  params=37,505  time=3.8s
k=10  val_loss=2.3177  ppl=10.15  params=45,825  time=4.3s
k=15  val_loss=2.3930  ppl=10.95  params=66,625  time=4.6s


## Section 4 — Validation loss and perplexity

**Perplexity** is the exponential of the average per-token cross-entropy loss:

```
perplexity = exp(val_loss)
```

A perplexity of P means the model is, on average, as uncertain as if it had to choose uniformly among P equally likely tokens. Lower is better.

The bigram model (Chapter 1) achieves perplexity ≈ 11 on tinyshakespeare. Watch what happens as k increases.


In [ ]:
# ── print a comparison table of loss and perplexity across all k values ───────
BAR = 30   # maximum character width of the ASCII bar chart column

print(f"{'k':>4}  {'val loss':>9}  {'perplexity':>11}  {'chart (lower = better)'}")
print(f"{'─'*4}  {'─'*9}  {'─'*11}  {'─'*BAR}")

# Find the best (lowest) perplexity across all trained models.
# We use this to scale bar lengths: the best model gets a full bar, others are shorter.
best_ppl = min(r['perplexity'] for r in results.values())

for k in K_VALUES:
    r = results[k]

    # Bar length formula: BAR * (best_ppl / this_ppl)
    # Best model  → ratio = 1.0 → full bar of width BAR.
    # Worse model → ratio < 1.0 → proportionally shorter bar.
    # chr(9608) is the Unicode full-block character █.
    bar = chr(9608) * int(BAR * best_ppl / r['perplexity'])
    print(f"{k:>4}  {r['val_loss']:>9.4f}  {r['perplexity']:>11.2f}  {bar}")

print()

# ── summarise the total improvement from k=1 to k=8 ──────────────────────────
k1 = results[1]['perplexity']
k8 = results[8]['perplexity']
# Relative improvement = (k1 - k8) / k1 * 100 — what fraction of k=1 perplexity was removed.
print(f"Perplexity reduction k=1 → k=8: {k1:.2f} → {k8:.2f}  "
      f"({100*(k1-k8)/k1:.1f}% relative improvement)")
print()

# ── theoretical worst case ────────────────────────────────────────────────────
# A model with NO information assigns equal probability (1/65) to all tokens.
# Its cross-entropy = log(65) ≈ 4.17 and its perplexity = 65.
# Any trained model should be well below this ceiling.
print("Entropy ceiling (uniform over vocab):",
      f"{vocab_size:.0f}  (perplexity if model assigns equal probability to all tokens)")


## Section 5 — Parameter count and training cost

More context means a wider input to the linear head — and more parameters. The cost grows **linearly** with k, not quadratically. That is a property of the concatenation architecture; we will see that attention has a different scaling profile (Chapter 7).


In [ ]:
# ── parameter count and training cost breakdown ───────────────────────────────
# GOAL: show that the embedding table is FIXED in size across all k,
# and only the linear head grows — linearly — as k increases.

print(f"{'k':>4}  {'params':>10}  {'head params':>12}  {'time (s)':>9}  {'ms/step':>8}")
print(f"{'─'*4}  {'─'*10}  {'─'*12}  {'─'*9}  {'─'*8}")

for k in K_VALUES:
    r = results[k]

    # EMBEDDING TABLE size: vocab_size × EMBED_DIM = 65 × 64 = 4,160.
    # This is IDENTICAL for every k — the embedding table depends only on vocab_size
    # and embed_dim, not on context length. It is the shared representation layer.
    emb_params = vocab_size * EMBED_DIM

    # HEAD PARAMETERS: total params minus the (fixed) embedding table.
    # head weights = k × EMBED_DIM × vocab_size  — grows linearly with k
    # head bias    = vocab_size                   — one bias per output logit
    # Together these are the only part that scales with context length.
    head_params = r['params'] - emb_params

    # Training time per step in milliseconds.
    # Larger head (bigger k) means a larger matrix multiply → slightly slower.
    ms_per = 1000 * r['train_sec'] / N_STEPS

    print(f"{k:>4}  {r['params']:>10,}  {head_params:>12,}  "
          f"{r['train_sec']:>9.1f}  {ms_per:>8.2f}")

print()
print("Embedding table is shared across all k (same vocab_size × embed_dim).")
print("Only the head grows: k × embed_dim × vocab_size extra weights per step.")
print()

# ── scaling factor: how much larger is k=8 vs k=1? ────────────────────────────
k1_params = results[1]['params']
k8_params = results[8]['params']
# ratio tells us how many times more parameters k=8 has than k=1.
print(f"k=1 params: {k1_params:,}   k=8 params: {k8_params:,}   "
      f"ratio: {k8_params/k1_params:.1f}×")


## Section 6 — Generated text at each k

Does the generated text improve as k increases? We generate 100 characters from each model starting from the same seed context.

With a very short context the model cannot respect word boundaries or sentence structure from earlier tokens. With longer context we expect smoother text — though it will remain far from coherent English until we add attention.


In [ ]:
# ── generate: produce text autoregressively from a trained model ──────────────
def generate(model, k, start_text, max_new=100, temperature=1.0):
    # AUTOREGRESSIVE GENERATION: predict one token at a time.
    # Each newly predicted token is immediately appended to the context for the next step.
    # The context window slides forward by one position after each generation step.

    model.eval()   # disable gradient tracking and any training-only behaviour

    # ── encode the seed text ──────────────────────────────────────────────────
    # Convert each character in start_text to its integer ID.
    # stoi.get(c, 0): if a character is not in the vocabulary, use ID 0 as a safe fallback.
    ids = [stoi.get(c, 0) for c in start_text]

    # ── pad if seed is shorter than k ─────────────────────────────────────────
    # The model always needs EXACTLY k input tokens — no more, no fewer.
    # If start_text has fewer than k characters, repeat the first token to pad.
    #
    # BIAS WARNING: this padding introduces a bias in the first k predictions.
    # The repeated token never appeared in training data — get_batch always
    # sampled positions with k real tokens to the left. After k steps the
    # padding tokens have all been replaced by genuinely generated tokens
    # and the model is back on familiar ground. For 100-character generation
    # runs the effect is negligible. The proper fix is a dedicated <PAD>
    # token trained into the vocabulary — not done here to keep things simple.
    while len(ids) < k:
        ids = [ids[0]] + ids

    # Start with the last k token IDs as the initial context window.
    context = ids[-k:]
    result  = []   # accumulates generated characters

    with torch.no_grad():   # no gradient computation needed during generation
        for _ in range(max_new):

            # STEP 1 — pack the context into a (1, k) tensor.
            # Batch size = 1: we generate one token at a time, one sequence at a time.
            x = torch.tensor([context], dtype=torch.long)   # shape: (1, k)

            # STEP 2 — forward pass through the model.
            # model(x) returns shape (1, vocab_size); [0] selects the single batch row.
            logits = model(x)[0]   # shape: (vocab_size,) = (65,)

            # STEP 3 — apply temperature and convert logits to probabilities.
            # Dividing by temperature reshapes the distribution before softmax:
            #   temperature < 1.0 → sharpens → model picks the most likely token more often
            #   temperature = 1.0 → standard softmax, no reshaping
            #   temperature > 1.0 → flattens → rarer tokens become more likely
            probs = F.softmax(logits / temperature, dim=-1)   # shape: (65,)

            # STEP 4 — sample one token from the probability distribution.
            # torch.multinomial: weighted random draw — tokens with higher probability
            # are more likely to be picked, but lower-probability tokens can win too.
            # This is NOT greedy (argmax) and NOT top-k/top-p:
            #   Greedy     → always picks the single highest probability token (deterministic)
            #   Top-k      → restricts the draw to the k highest probability tokens only
            #   Top-p      → restricts the draw to the smallest set summing to probability p
            #   Here       → plain multinomial: all 65 tokens are eligible, weighted by prob
            # Top-k and top-p were covered in Chapter 5. Plain multinomial is used here
            # to keep the focus on context length k, not sampling strategy.
            nxt = torch.multinomial(probs, num_samples=1).item()   # integer token ID

            # STEP 5 — decode and store the generated character.
            result.append(itos[nxt])

            # STEP 6 — slide the context window forward by one position.
            # context[1:]  removes the oldest token from the left end of the window.
            # + [nxt]      appends the newly generated token to the right end.
            # The window always contains exactly k tokens — ready for the next iteration.
            context = context[1:] + [nxt]

    return ''.join(result)   # join all generated characters into a single string

# ── generate_topk: same as generate but restricts sampling to the top-k tokens ─
def generate_topk(model, k, start_text, top_k=50, max_new=100, temperature=1.0):
    # TOP-K SAMPLING: before drawing from the probability distribution, zero out
    # all tokens except the top_k highest-probability ones. Then renormalise and draw.
    #
    # WHY: plain multinomial can occasionally pick very low-probability tokens
    # (e.g. probability 0.001) which produces surprising/incoherent characters.
    # Restricting to the top 50 eliminates the long tail of unlikely tokens while
    # still preserving randomness among the plausible candidates.
    #
    # CONTRAST WITH GREEDY: greedy always picks rank-1. top-k=50 picks randomly
    # among ranks 1-50. top-k=1 IS greedy.

    model.eval()
    ids = [stoi.get(c, 0) for c in start_text]
    while len(ids) < k:
        ids = [ids[0]] + ids
    context = ids[-k:]
    result  = []

    with torch.no_grad():
        for _ in range(max_new):
            x      = torch.tensor([context])
            logits = model(x)[0]                              # shape: (65,)
            probs  = F.softmax(logits / temperature, dim=-1)  # shape: (65,)

            # ── top-k filter ──────────────────────────────────────────────────
            # torch.topk returns the top_k largest values and their indices.
            # We zero out everything below the top_k threshold, then renormalise.
            top_probs, top_indices = torch.topk(probs, min(top_k, probs.size(-1)))
            filtered = torch.zeros_like(probs)            # start with all zeros
            filtered[top_indices] = top_probs             # fill in only the top-k
            filtered = filtered / filtered.sum()          # renormalise to sum to 1

            nxt = torch.multinomial(filtered, num_samples=1).item()
            result.append(itos[nxt])
            context = context[1:] + [nxt]

    return ''.join(result)

# ── compare plain multinomial vs top-k=50 on k=8 and k=20 ────────────────────
# Same seed, same random seed — only the sampling strategy differs.
# This isolates the effect of top-k filtering from the effect of context length.
SEED = "First Citizen:
"

print("── Plain multinomial (all 65 tokens eligible) ───────────────────────────")
torch.manual_seed(42)
for k in K_VALUES:
    sample = generate(results[k]['model'], k, SEED)
    print(f"k={k}:  {repr(sample)}")
print()

print("── Top-k sampling (top 50 tokens only) ─────────────────────────────────")
torch.manual_seed(42)
for k in K_VALUES:
    sample = generate_topk(results[k]['model'], k, SEED, top_k=50)
    print(f"k={k}:  {repr(sample)}")
print()

print("Note: with only 65 tokens in the vocab, top-k=50 still allows most tokens.")
print("The effect of top-k is most visible at low k where the distribution is flat.")
print("At high k the model already concentrates probability on a few tokens,")
print("so top-k=50 changes little. Top-k matters most on large (50k+) vocabularies.")


---

## Part 3 — What the Model Learns

We now look inside the improvement: which characters benefit most from longer context, how the neural model compares to a count-based n-gram, and what the architecture fundamentally cannot do.


## Section 7 — The fixed-position limitation

The `FixedWindowModel` concatenates position-specific embeddings and feeds them to a single linear layer. The first `embed_dim` columns of the weight matrix always respond to position −k (oldest); the last `embed_dim` columns always respond to position −1 (most recent).

This means the model treats positions as **fixed slots**, not as content-bearing tokens whose relevance depends on what they contain.

Consider the phrase "the bank of":

- After "bank", knowing whether the earlier word was "river" or "money" would   change the predicted next token substantially.
- But the fixed-window model assigns the same weight to position −3 regardless   of whether that slot contains "river" or "money".

The model has no mechanism to say "this earlier position matters more for this particular prediction". Every position always contributes with the same learned weight.

> **Key limitation:** Context window is fixed, and positions are combined in a predetermined way. The model cannot learn which positions matter — it can only learn which positions *tend* to matter on average across the corpus.

This is what self-attention (Chapter 7) is designed to address: it computes **content-dependent** weights over context positions instead of fixed-slot weights.


In [ ]:
# ── inspect the head weight matrix to expose the fixed-position structure ──────
#
# The head weight matrix W has shape (vocab_size, k * embed_dim) = (65, 256) for k=4.
#
# STRUCTURE: each ROW of W corresponds to one output vocabulary token.
# Each row is divided into k blocks of embed_dim columns — one block per context position:
#
#   Block 0  cols   0.. 63  — weights that ALWAYS respond to position −4 (oldest token)
#   Block 1  cols  64..127  — weights that ALWAYS respond to position −3
#   Block 2  cols 128..191  — weights that ALWAYS respond to position −2
#   Block 3  cols 192..255  — weights that ALWAYS respond to position −1 (newest token)
#
# KEY QUESTION: does the model learn to favour some positions over others?
# We measure the average L2 norm of each block across all 65 output tokens.
# Similar norms across blocks → the model treats all positions with similar importance.
# This confirms the limitation: no content-based reweighting is possible.

model_k4 = results[4]['model']

# head.weight has shape (vocab_size, k * embed_dim) = (65, 256).
# .detach() removes the tensor from the computation graph — we only want to read values.
W = model_k4.head.weight.detach()
print(f"Head weight shape: {list(W.shape)}")
print(f"  = (vocab_size={vocab_size}, k * embed_dim = 4 × {EMBED_DIM})")
print()

k4    = 4
block = EMBED_DIM   # width of each positional block in columns

norms_by_pos = []
for pos in range(k4):
    # STEP 1 — slice the columns belonging to this context position.
    # W[:, pos*block : (pos+1)*block] selects ALL rows (all output tokens)
    # but only the embed_dim columns for this position.
    # Result shape: (vocab_size, embed_dim) = (65, 64).
    block_W = W[:, pos * block : (pos + 1) * block]

    # STEP 2 — compute the L2 norm of each row in this block.
    # norm(dim=1): for each of the 65 output tokens, compute sqrt(sum of squares)
    # across the 64 embedding dimensions.
    # A high norm for a given output token means this position strongly influences
    # the prediction of that token.
    # Result shape: (vocab_size,) = (65,).
    pos_norm = block_W.norm(dim=1)

    # STEP 3 — average across all 65 output tokens to get one number per position.
    norms_by_pos.append(pos_norm.mean().item())

print("Average L2 weight norm per context position (k=4 model):")
print("Position −4 is oldest; position −1 is most recent.")
print()
for pos in range(k4):
    label = f"pos -{k4 - pos}"
    bar   = chr(9608) * int(norms_by_pos[pos] * 300)
    print(f"  {label}  {norms_by_pos[pos]:.4f}  {bar}")

print()
# INTERPRETATION:
# Roughly equal norms suggest the model has not learned a global preference
# for recent over distant positions on average. This is one diagnostic lens —
# norms alone do not fully measure importance (embedding directions and
# correlations also matter), but the result is consistent with the additive
# independent structure of the model.
#
# THE PRECISE LIMITATION:
# Each position's contribution = W_block · embedding(token at that position).
# The embedding DOES change with the token — the model is not blind to identity.
# What the model cannot do: let position j's transformation adapt based on
# what token is at position i. Every position applies its own fixed learned
# transformation independently. The contributions are then summed.
#
# Self-attention replaces these fixed independent transformations with
# content-based similarity scores: score(t,s) = query[t] · key[s],
# computed fresh for every input — enabling cross-position interaction.
print("Norms are roughly equal — no global positional preference on average.")
print("(This is a diagnostic, not a complete measure of positional importance.)")
print()
print("The precise limitation: each position applies a fixed learned transformation")
print("of its own token. That transformation cannot adapt based on other positions.")
print()
print("Self-attention (Ch7) computes score(t,s) = query[t] * key[s] — dynamic,")
print("content-based routing across positions, computed fresh for every input.")


## When does more context help?

| Context length | What the model gains |
|---|---|
| k = 1 | Previous character only — useful for spelling patterns |
| k = 2–3 | Short multi-character sequences — common digrams/trigrams |
| k = 4–6 | Short words and word endings — local morphology |
| k = 8+ | Cross-word patterns — early syntax, phrase-level structure |

The validation loss improvement has **diminishing returns**: each doubling of k buys less perplexity reduction than the previous doubling. Meanwhile parameter count and training cost grow linearly.

The n-gram bridge shows why: a count-based n-gram already captures most of the short-range statistical structure. The neural model's advantage is parameter efficiency and smooth generalisation to unseen contexts — not fundamentally different information.

> **Chapter 6 limitation.** Adding more context always helps *on average*, but the model cannot selectively attend to the most relevant positions. A fixed-slot concatenation treats position −8 as equally important as position −1 for every prediction. The next chapter replaces this with a mechanism that computes context-position importance from the tokens themselves.


---

## Lab — Exercises

Each exercise builds on the trained models in `results`. Solutions should run in the same notebook without re-training.


### Exercise 1 — Train k = 12

Train a `FixedWindowModel` with k = 12. Does validation loss continue to decrease, or do you see diminishing returns compared to k = 8? Report the perplexity and plot it alongside the k = 1..8 results.


In [ ]:
# ── Exercise 1: does k=12 improve over k=8? ───────────────────────────────────
# QUESTION: does adding more context beyond k=8 continue to reduce perplexity,
# or do we hit diminishing returns?
#
# We train a new FixedWindowModel with k=12 using the SAME hyperparameters
# (N_STEPS, BATCH_SIZE, LR) as the main training loop for a fair comparison.

torch.manual_seed(42)   # same seed for reproducibility
k12 = 12
m12 = FixedWindowModel(vocab_size, k12)   # fresh model, random weights
opt = torch.optim.AdamW(m12.parameters(), lr=LR)

for step in range(N_STEPS):
    m12.train()
    x, y = get_batch(train_data, k12, BATCH_SIZE)
    loss = F.cross_entropy(m12(x), y)
    opt.zero_grad(); loss.backward(); opt.step()

# Evaluate on 2048 val samples for a stable final estimate — same as main loop.
vl12  = eval_loss(m12, val_data, k12, n_samples=2048)
ppl12 = torch.exp(torch.tensor(vl12)).item()   # perplexity = exp(val_loss)
print(f"k=12  val_loss={vl12:.4f}  perplexity={ppl12:.2f}")
print(f"k= 8  val_loss={results[8]['val_loss']:.4f}  perplexity={results[8]['perplexity']:.2f}")


### Exercise 2 — Loss curves

Each model in `results` has a `loss_curve` list of `(step, val_loss)` pairs recorded every 500 steps. Print a side-by-side ASCII loss curve for k = 1, 4, 8 showing how quickly each model converges.


In [ ]:
# ── Exercise 2: compare convergence speed across k=1, k=4, k=8 ───────────────
# QUESTION: do higher-k models converge faster, slower, or similarly?
#
# During training, eval_loss was recorded every 500 steps in results[k]['loss_curve'].
# loss_curve is a list of (step, val_loss) tuples — one entry per 500 steps.
# Printing them side by side lets us compare convergence trajectories.

print(f"{'step':>6}  {'k=1':>8}  {'k=4':>8}  {'k=8':>8}")
print(f"{'─'*6}  {'─'*8}  {'─'*8}  {'─'*8}")

# All three loss_curve lists have the same number of entries (N_STEPS // 500 = 6).
# We iterate by index i to read the checkpoint at the same training step for all three.
for i, (step, _) in enumerate(results[1]['loss_curve']):
    v1 = results[1]['loss_curve'][i][1]   # val_loss for k=1 at this checkpoint
    v4 = results[4]['loss_curve'][i][1]   # val_loss for k=4 at this checkpoint
    v8 = results[8]['loss_curve'][i][1]   # val_loss for k=8 at this checkpoint
    print(f"{step:>6}  {v1:>8.4f}  {v4:>8.4f}  {v8:>8.4f}")


### Exercise 3 — Characters that benefit most from context

Using the `char_surprisal` function and `surp_k1` / `surp_k8` already computed in Section 7, find the 5 characters with the **smallest** gain (least benefit from longer context). Why might those characters be predictable from k=1 alone?


In [ ]:
# ── Exercise 3: which characters benefit LEAST from more context? ─────────────
# QUESTION: are there characters that are just as predictable from k=1 as from k=8?
#
# surp_k1, surp_k8, and gain were computed in Section 7.
# gain[ch] = surp_k1[ch] - surp_k8[ch] — bits saved by going from k=1 to k=8.
#
# Characters with SMALL gain are already well-predicted from just one preceding token.
# They tend to follow highly predictable local rules:
#   e.g. 'u' almost always follows 'q', or a digit follows another digit.

# Sort gain ascending — smallest gain (fewest bits saved) at the top.
low_gainers = sorted(gain.items(), key=lambda x: x[1])[:10]

print(f"{'char':>6}  {'surp k=1':>9}  {'surp k=8':>9}  {'gain':>7}")
print(f"{'─'*6}  {'─'*9}  {'─'*9}  {'─'*7}")
for ch, g in low_gainers:
    print(f"{repr(ch):>6}  {surp_k1[ch]:>9.3f}  {surp_k8[ch]:>9.3f}  {g:>7.3f}")


### Exercise 4 — Varying embed_dim

Re-train `FixedWindowModel` with k = 4 and `embed_dim = 32` and `embed_dim = 128`. Compare validation loss and parameter count to the default `embed_dim = 64`. Is the model more sensitive to k or to embed_dim?


In [ ]:
# ── Exercise 4: is the model more sensitive to k or to embed_dim? ─────────────
# QUESTION: which matters more — how many context tokens we use (k),
# or how large the internal representation of each token is (embed_dim)?
#
# We hold k constant at 4 and vary embed_dim: 32 (half default), 64 (default), 128 (double).
# Fixing k isolates the effect of embed_dim — any difference in val_loss comes from
# representation capacity, not from the amount of context the model can see.

for edim in [32, 64, 128]:
    torch.manual_seed(42)   # same random init for all three — only embed_dim differs
    m   = FixedWindowModel(vocab_size, 4, embed_dim=edim)
    opt = torch.optim.AdamW(m.parameters(), lr=LR)

    for step in range(N_STEPS):
        m.train()
        x, y = get_batch(train_data, 4, BATCH_SIZE)
        loss = F.cross_entropy(m(x), y)
        opt.zero_grad(); loss.backward(); opt.step()

    vl = eval_loss(m, val_data, 4, n_samples=2048)
    # Count all learnable parameters: embedding table (vocab*edim) + head weights + bias.
    p  = sum(pp.numel() for pp in m.parameters())
    print(f"embed_dim={edim:>3}  val_loss={vl:.4f}  params={p:,}")


### Exercise 5 — Count bigram accuracy

For each of the 65 vocabulary tokens as target, compute the fraction of positions in `val_data` where the count bigram (k=1) correctly predicts the top token. Then do the same for the neural k=1 model. Are they similar? Why or why not?


In [ ]:
# ── Exercise 5: top-1 accuracy — count bigram vs neural k=1 ──────────────────
# QUESTION: does a neural k=1 model predict the correct next character more often
# than a count bigram that has seen the same amount of training data?
#
# TOP-1 ACCURACY = fraction of positions where the model's single best guess is correct.
# This is stricter than perplexity:
#   Perplexity rewards the model for assigning any probability mass to the correct token.
#   Top-1 accuracy requires the correct token to be the SINGLE highest-scored prediction.
#
# Note: character-level accuracy is typically low because many positions are genuinely
# ambiguous (many valid next characters after 'th', for example). Perplexity is the
# standard quality metric; top-1 accuracy is included here for intuition.

def top1_accuracy(model, k, data, n=2000):
    # Evaluate over n consecutive positions starting at position k in data.
    correct = 0
    model.eval()
    with torch.no_grad():
        for t in range(k, k + n):
            # Build a (1, k) context tensor from the k tokens before position t.
            ctx    = data[t - k : t].unsqueeze(0)   # shape: (1, k)
            target = data[t].item()                 # correct next token ID

            # argmax selects the index with the highest logit — the greedy best guess.
            # .item() converts the 0-D tensor to a plain Python integer.
            pred = model(ctx).argmax(dim=-1).item()
            correct += int(pred == target)   # 1 if correct, 0 otherwise
    return correct / n   # fraction correct, range 0.0 to 1.0

def count_top1_accuracy(counts, k, data, n=2000):
    # Same evaluation for a count n-gram model.
    correct = 0
    for t in range(k, k + n):
        ctx    = tuple(data[t - k : t].tolist())   # hashable tuple for dict lookup
        target = data[t].item()
        dist   = counts.get(ctx, {})
        if dist:
            # max(dist, key=dist.get) returns the token ID with the highest raw count.
            pred = max(dist, key=dist.get)
            correct += int(pred == target)
    return correct / n

neural_acc = top1_accuracy(results[1]['model'], 1, val_data)
count_acc  = count_top1_accuracy(bigram_counts, 1, val_data)
print(f"Neural k=1  top-1 accuracy: {neural_acc:.3f}")
print(f"Count bigram top-1 accuracy: {count_acc:.3f}")
print()
print("Note: accuracy on character-level data is low because many positions")
print("are genuinely ambiguous — perplexity is a better model quality metric.")


### Exercise 6 — block_size vs k: are they the same thing?

The Chapter 1 `BigramModel` was trained with `block_size = 8`, yet its effective context
length is 1. The Chapter 6 `FixedWindowModel` trained with `k = 8` also uses 8 tokens —
but for a completely different reason.

**Task A** — Run the cell below. It re-creates the Chapter 1 `get_batch` and shows what
`x` and `y` look like for the bigram model.

**Task B** — Explain in words: why does the Chapter 1 model pack 8 positions per window
even though the BigramModel only uses 1 token per prediction?

**Task C** — What would you have to change in the `BigramModel.forward()` to make it
actually use all 8 tokens? (You do not need to implement it — just describe the change.)


In [ ]:
# ── Exercise 6: block_size vs k ──────────────────────────────────────────────
#
# BACKGROUND:
#   Chapter 1 BigramModel: block_size=8, but effective context = 1.
#   Chapter 6 FixedWindowModel: k=8, effective context = 8.
#
# The difference: the BigramModel's embedding_table(idx) processes EACH POSITION
# independently. It never mixes information across positions. block_size is only
# a training efficiency trick — 8 (context, target) pairs packed into one window.
#
# The FixedWindowModel flattens all k embeddings into one vector before the head,
# so the head CAN mix information across positions. k is the true context length.

# ── re-create Chapter 1 get_batch ────────────────────────────────────────────
block_size_ch1 = 8
batch_size_ch1 = 4

def get_batch_ch1(data, block_size, batch_size):
    # Sample batch_size random start positions.
    # Upper bound = len(data) - block_size so x and y both fit.
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # x: tokens at positions i .. i+block_size-1   — the context window
    # y: tokens at positions i+1 .. i+block_size   — all shifted by 1
    # Both have shape (batch_size, block_size).
    # This packs block_size predictions into one window:
    #   position 0: given x[0]           → predict y[0]
    #   position 1: given x[1]           → predict y[1]
    #   ...
    #   position 7: given x[7]           → predict y[7]
    # BUT the BigramModel treats each position independently — x[0] does NOT
    # influence the prediction at position 1. It only sees its own token.
    x = torch.stack([data[i     : i + block_size    ] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y

torch.manual_seed(42)
xb, yb = get_batch_ch1(val_data, block_size_ch1, batch_size_ch1)

print("Chapter 1 batch — shape:", list(xb.shape), "(batch, block_size)")
print()
print("Each ROW is one training window. Each COLUMN is one (context, target) pair.")
print()
for b in range(batch_size_ch1):
    row_x = [itos[i.item()] for i in xb[b]]
    row_y = [itos[i.item()] for i in yb[b]]
    print(f"  window {b}: x = {''.join(row_x)!r}  →  y = {''.join(row_y)!r}")

print()
print("The BigramModel sees each position independently:")
print("  position 0: given", repr(itos[xb[0,0].item()]), "→ predict", repr(itos[yb[0,0].item()]))
print("  position 1: given", repr(itos[xb[0,1].item()]), "→ predict", repr(itos[yb[0,1].item()]))
print("  ...and so on — no cross-position mixing.")
print()
print("Chapter 6 FixedWindowModel with k=8:")
print("  given ALL 8 tokens together → predict 1 target")
print("  (all 8 are flattened into one vector before the head)")
print()
print("Summary:")
print("  block_size in Ch1 = training efficiency trick (8 preds per window, model uses 1)")
print("  k in Ch6          = true context length (model uses all k tokens)")


### Exercise 7 — Per-character surprisal (research connection)

This exercise connects Chapter 6 back to Paper 1 ("Which Characters Need Context?").

Paper 1 measured **Context Gain** using n-gram counts: how many bits does knowing
the preceding character save when predicting character X?

Here we measure the same thing using neural models: how many bits does going from
k=1 to k=8 save per character?

**Task A** — Run the cell below. Which characters show the largest surprisal gain?

**Task B** — Compare your results to Paper 1's Context Gain table. Do the same
characters rank highly in both? If not, why might they differ?

**Task C** — Which characters show near-zero gain? What do they have in common?


In [ ]:
# ── per-character surprisal analysis ─────────────────────────────────────────
#
# SURPRISAL at position t = -log2( p(correct token | k preceding tokens) )
# Units: bits. Intuition:
#   High surprisal → the model was wrong — it needed many bits to encode this token.
#   Low surprisal  → the model was confident and correct.
#
# By averaging surprisal over every position where a given character is the TARGET,
# we get the mean surprisal for that character under a model with context length k.
#
# GAIN = surp(k=1) - surp(k=8)
# A large positive gain means the character is much easier to predict with 8 context
# tokens than with 1. Near-zero gain means additional context does not help.
# This is the neural analogue of Context Gain from Paper 1 (the Chapter 1 research paper).

TEST_LEN = 2000   # number of positions to evaluate (first 2000 positions of val_data)
MAX_K    = 8      # we need MAX_K tokens of prefix so the k=8 model can form its first window
# Include enough leading tokens so k=8 can build its first context window at position 0.
test_seg = val_data[:TEST_LEN + MAX_K]

def char_surprisal(model, k, data):
    # Returns: {character: mean surprisal in bits} over all positions in data.
    model.eval()
    totals = {}   # character → cumulative sum of surprisal across all its occurrences
    counts = {}   # character → number of occurrences seen

    with torch.no_grad():
        for t in range(k, len(data) - 1):

            # STEP 1 — build the context tensor for position t.
            # data[t-k:t] = the k tokens immediately before position t (shape: (k,)).
            # unsqueeze(0) adds the batch dimension: (k,) → (1, k).
            ctx    = data[t - k : t].unsqueeze(0)   # shape: (1, k)
            target = data[t].item()                 # the correct next token ID

            # STEP 2 — forward pass, then compute log-probabilities.
            # F.log_softmax is numerically more stable than torch.log(F.softmax(...)).
            # It returns log(p) for each vocab token — shape: (vocab_size,).
            logits = model(ctx)[0]
            log_p  = F.log_softmax(logits, dim=-1)

            # STEP 3 — compute surprisal in bits for the correct token.
            # log_p[target] = log(p) in nats (natural log base e).
            # Dividing by log(2) = 0.6931 converts from nats to bits.
            # We negate because surprisal = -log2(p) and log_p is already negative.
            surprisal = -(log_p[target].item() / 0.6931)

            # Accumulate surprisal for this character.
            ch = itos[target]
            totals[ch] = totals.get(ch, 0.0) + surprisal
            counts[ch] = counts.get(ch, 0) + 1

    # Divide cumulative surprisal by count to get the MEAN surprisal per character.
    return {ch: totals[ch] / counts[ch] for ch in totals}

# ── compute mean surprisal under k=1 and k=8 ─────────────────────────────────
surp_k1 = char_surprisal(results[1]['model'], 1, test_seg)
surp_k8 = char_surprisal(results[8]['model'], 8, test_seg)

# ── compute gain: bits saved by going from k=1 to k=8 per character ──────────
# Positive gain → the character benefits from more context.
# Near-zero gain → the character is already predictable from just 1 preceding token.
gain = {ch: surp_k1[ch] - surp_k8.get(ch, surp_k1[ch])
        for ch in surp_k1 if ch in surp_k8}

# Sort descending — characters with the largest gain (most benefit) come first.
top_gainers = sorted(gain.items(), key=lambda x: -x[1])[:15]

print(f"{'char':>6}  {'surp k=1':>9}  {'surp k=8':>9}  {'gain':>7}  (bits)")
print(f"{'─'*6}  {'─'*9}  {'─'*9}  {'─'*7}")
for ch, g in top_gainers:
    # Scale bar: 1 bit of gain maps to 6 block characters wide.
    bar = chr(9608) * int(g * 6)
    print(f"{repr(ch):>6}  {surp_k1[ch]:>9.3f}  {surp_k8[ch]:>9.3f}  "
          f"{g:>7.3f}  {bar}")

print()
print("Characters with the largest gain are those whose identity is most")
print("predictable from longer preceding context — structural and lexical patterns.")


### Exercise 8 — The n-gram bridge

A count-based n-gram stores observed character frequencies directly.
A neural fixed-window model compresses those same frequencies into a weight matrix.
Both are approximating the same thing: `p(next token | k preceding tokens)`.

**Task A** — Run the cell below. For the same context, compare the top-3 predictions
from the count bigram vs the neural k=1 model. Do they agree?

**Task B** — Compare count trigram vs neural k=2. Does the neural model match
the count model more or less closely than in Task A?

**Task C** — Find a context where the neural model and count model strongly disagree.
Why might they differ? (Hint: think about unseen contexts.)


In [ ]:
from collections import defaultdict

# ── build_ngram: create a count-based n-gram model ───────────────────────────
def build_ngram(data, k):
    # PURPOSE: count how often each k-token context is followed by each next token.
    #
    # RETURN VALUE: nested defaultdict  {context_tuple: {next_token_id: count}}
    #   Outer key: tuple of k integer token IDs (the context).
    #              We use a TUPLE (not a list) because tuples are hashable — lists are not,
    #              and dict keys must be hashable.
    #   Inner key: integer token ID of the token that followed this context.
    #   Value:     how many times this (context, next_token) pair appeared in the data.
    #
    # defaultdict(lambda: defaultdict(int)):
    #   Outer defaultdict: accessing a missing context key creates an empty inner dict.
    #   Inner defaultdict(int): accessing a missing next_token key returns 0.
    #   This avoids KeyError exceptions and makes the counting loop clean.
    counts = defaultdict(lambda: defaultdict(int))
    for t in range(k, len(data)):
        # context = the k tokens before position t, as a hashable tuple.
        ctx = tuple(data[t - k : t].tolist())
        # next token = the token at position t.
        nxt = data[t].item()
        counts[ctx][nxt] += 1
    return counts

# Build a bigram (k=1) and trigram (k=2) count table from training data.
bigram_counts  = build_ngram(train_data, 1)
trigram_counts = build_ngram(train_data, 2)

# ── ngram_top3: look up the top-3 predictions from a count n-gram ─────────────
def ngram_top3(counts, context_ids):
    # Convert the context list to a tuple for dict lookup.
    ctx  = tuple(context_ids)
    # counts.get(ctx, {}) returns an empty dict if this context was NEVER seen in training.
    dist = counts.get(ctx, {})
    if not dist:
        return [('?', 0.0)]   # unseen context — the count model cannot make a prediction
    total = sum(dist.values())   # total times this context appeared across all successors
    top   = sorted(dist.items(), key=lambda x: -x[1])[:3]   # top-3 (token_id, count) pairs
    # Convert raw counts to probabilities: count / total_count_for_this_context.
    return [(itos[i], c / total) for i, c in top]

# ── neural_top3: look up top-3 predictions from a trained neural model ────────
def neural_top3(model, k, context_ids):
    # Use the last k elements of context_ids as the model input (handles len > k).
    x      = torch.tensor([context_ids[-k:]])   # shape: (1, k)
    logits = model(x)[0]                        # shape: (vocab_size,)
    probs  = F.softmax(logits, dim=-1)          # convert raw scores to probabilities
    top    = torch.topk(probs, 3)               # top-3 (probability, index) pairs
    return [(itos[i.item()], p.item()) for p, i in zip(top.values, top.indices)]

# ── compare on five representative contexts ───────────────────────────────────
test_contexts = [
    ("'e'",  [stoi['e']]),             # single character — common vowel
    ("' '",  [stoi[' ']]),             # single character — space (word boundary)
    ("'th'", [stoi['t'], stoi['h']]),  # two characters — very common English digram
    ("'e '", [stoi['e'], stoi[' ']]),  # two characters — end of a word
    ("'ng'", [stoi['n'], stoi['g']]),  # two characters — common suffix (-ing)
]

for label, ctx_ids in test_contexts:
    bg = ngram_top3(bigram_counts,  ctx_ids[-1:])   # count bigram: uses last 1 token only
    tg = ngram_top3(trigram_counts, ctx_ids[-2:])   # count trigram: uses last 2 tokens
    n1 = neural_top3(results[1]['model'], 1, ctx_ids)   # neural k=1: same info as count bigram
    n2 = neural_top3(results[2]['model'], 2, ctx_ids)   # neural k=2: same info as count trigram
    print(f"Context {label}:")
    print(f"  count bigram  : {bg}")
    print(f"  count trigram : {tg}")
    print(f"  neural k=1    : {n1}")
    print(f"  neural k=2    : {n2}")
    print()


### Exercise 9 — The XOR test: what a linear fixed-window model cannot represent

This exercise builds the decisive experiment from the chapter review.

**The claim:** the `FixedWindowModel` combines information from all k positions — but
each position applies a fixed transformation of its own token independently. The sum
of independent linear transformations cannot represent every possible relationship
among inputs. Specifically, it cannot represent XOR.

**XOR:** given two binary tokens A and B, predict 1 if exactly one of them is 1,
else predict 0. The answer depends on the interaction between A and B — neither
token alone determines the output.

**Task A** — Run the cell below. It trains a linear fixed-window model (k=2) on
all four XOR examples. Does it learn XOR perfectly?

**Task B** — Uncomment the nonlinear model (one hidden layer with ReLU). Does
the nonlinear model learn XOR?

**Task C** — What does this tell you about what attention needs to add beyond
what a linear fixed-window model already has?


In [ ]:
# ── Exercise 9: XOR test — what a linear fixed-window model cannot represent ──
#
# XOR truth table:
#   input [0, 0] → target 0
#   input [0, 1] → target 1
#   input [1, 0] → target 1
#   input [1, 1] → target 0
#
# The correct answer depends on the INTERACTION between both positions.
# Neither position alone determines the output.
# A linear model computes: score = W_left · embed(A) + W_right · embed(B)
# This is a sum of two independent terms — it cannot represent XOR exactly.

import torch
import torch.nn as nn
import torch.nn.functional as F

# ── synthetic XOR dataset ─────────────────────────────────────────────────────
# Vocabulary: {0, 1} — binary tokens.
# All four (context, target) pairs from the XOR truth table.
# We repeat them 100x to give the model enough gradient signal.
x_xor = torch.tensor([[0,0],[0,1],[1,0],[1,1]]).repeat(100, 1)  # (400, 2)
y_xor = torch.tensor([0, 1, 1, 0]).repeat(100)                  # (400,)

# ── linear fixed-window model (k=2, vocab=2, embed_dim=4) ────────────────────
# This is structurally identical to FixedWindowModel but on a 2-token vocabulary.
# The head is purely linear: score = W · [embed(A) || embed(B)] + b
# Each position contributes independently — no cross-position interaction.
class LinearXOR(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(2, 4)          # 2 tokens, embed_dim=4
        self.head  = nn.Linear(2 * 4, 2)         # k=2, output 2 classes

    def forward(self, x):
        emb = self.embed(x)                      # (batch, 2, 4)
        emb = emb.view(emb.size(0), -1)          # (batch, 8)
        return self.head(emb)                    # (batch, 2)

# ── nonlinear model — uncomment to test ───────────────────────────────────────
# A single hidden layer with ReLU allows cross-position interactions.
# class NonlinearXOR(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.embed  = nn.Embedding(2, 4)
#         self.hidden = nn.Linear(2 * 4, 8)
#         self.head   = nn.Linear(8, 2)
#     def forward(self, x):
#         emb = self.embed(x).view(x.size(0), -1)
#         return self.head(F.relu(self.hidden(emb)))

# ── train the linear model ────────────────────────────────────────────────────
torch.manual_seed(42)
model = LinearXOR()
opt   = torch.optim.Adam(model.parameters(), lr=1e-2)

for step in range(2000):
    model.train()
    loss = F.cross_entropy(model(x_xor), y_xor)
    opt.zero_grad(); loss.backward(); opt.step()

# ── evaluate ──────────────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    preds = model(torch.tensor([[0,0],[0,1],[1,0],[1,1]])).argmax(dim=1)

print("XOR truth table — linear fixed-window model:")
print(f"  [0,0] → predicted {preds[0].item()}  (correct: 0)")
print(f"  [0,1] → predicted {preds[1].item()}  (correct: 1)")
print(f"  [1,0] → predicted {preds[2].item()}  (correct: 1)")
print(f"  [1,1] → predicted {preds[3].item()}  (correct: 0)")
print()

correct = (preds == torch.tensor([0,1,1,0])).sum().item()
print(f"Correct: {correct}/4")
print()
print("A linear model scores each token independently and sums the results.")
print("XOR requires knowing BOTH tokens together — no linear decomposition can")
print("represent this. The model will get at most 3/4 correct.")
print()
print("Uncomment NonlinearXOR above and re-run to see a model that CAN learn XOR.")
print("Attention's advantage over a simple hidden layer: it constructs cross-position")
print("interactions with shared projections (Q,K,V matrices) that generalise across")
print("all position pairs, rather than one fixed transformation per slot.")


## Research Connections

**Bengio et al. (2003)** — *A Neural Probabilistic Language Model*
The paper that introduced the feedforward neural language model: embed tokens, concatenate, project to a prediction. Chapter 6's `FixedWindowModel` is a direct implementation of this architecture at character level. Bengio's model used a hidden layer with tanh nonlinearity; ours uses a single linear head. The key insight was that the embedding table allows generalisation across similar contexts — something count n-grams cannot do.

**Mikolov et al. (2013)** — *Distributed Representations of Words and Phrases*
Showed that learned token embeddings develop geometric structure reflecting semantic and syntactic relationships. The embedding table in `FixedWindowModel` learns a similar (though character-level) representation: characters that appear in similar contexts end up with similar vectors.

**Paper 1 — *Which Characters Need Context?***
The per-character surprisal analysis in Section 7 is a neural analogue of the Context Gain metric from Paper 1. Where Paper 1 measured context gain in a count n-gram model across multiple corpora, the same question can now be asked of a neural fixed-window model: do structural characters (punctuation) show larger surprisal reduction than lexical characters as k increases?

**Research bridge (Chapter 6)**
> Do n-gram-derived character-specific context requirements predict which targets benefit from additional context in a neural fixed-window language model?

Candidate quantities: n-gram Context Gain, neural surprisal gain, coverage / sparsity of the n-gram counts, corpus differences (prose vs code). This question has not been answered in the literature and is a natural continuation of Paper 1.


## What This Chapter Deliberately Does Not Explain

**Hidden layers and nonlinearities**
Bengio et al. (2003) added a `tanh` hidden layer between the concatenated embeddings and the output head. This can improve model capacity but complicates the parameter-count story. Chapter 6 uses a single linear head so the fixed-position structure is easy to inspect.

**Recurrent neural networks (RNNs)**
An RNN processes tokens sequentially and maintains a hidden state — effectively an unbounded context window. RNNs predate the fixed-window approach and solve the truncation problem differently, but they have their own limitations (vanishing gradients, sequential compute). They are not on the book's critical path.

**Padding and masking**
When a sequence is shorter than k, production systems pad with a special token and mask the padded positions. Chapter 6 avoids sequences shorter than k by sampling only from positions t ≥ k.

**Byte-pair encoding (BPE) with context windows**
The Chapter 6 model operates at character level (vocab = 65). BPE tokenisers (Chapter 4) change the effective context span: k = 8 subword tokens covers much more text than k = 8 characters. The interaction between tokenisation granularity and context window size is a non-trivial design decision in production models.

**Learning rate scheduling**
All models here use a flat learning rate of 1e-3. Cosine decay or warmup schedules typically improve final performance. Chapter 6 holds lr constant to isolate the effect of k.
